In [ ]:
import cv2
import pytesseract
from pytesseract import Output
import numpy as np
import os
import csv
import json
from skimage.metrics import structural_similarity as ssim
from datetime import timedelta

# ---------------- CONFIG ----------------
VIDEO_FILE = "2026-05-13-graduate.mp4"

START_TIME = "00:38:02"
END_TIME   = "01:12:02"

ANNOTATED_DIR = "annotated"
CSV_FILE = "ocr_results.csv"
JSON_FILE = "ocr_results.json"

os.makedirs(ANNOTATED_DIR, exist_ok=True)

# OCR ROI
X0, Y0 = 250, 520
X1, Y1 = 1920, 1080

# OCR config
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
TESS_CFG = "--psm 6 --oem 3 -l eng"

def ts_to_seconds(ts):
    h, m, s = ts.split(":")
    return int(h)*3600 + int(m)*60 + float(s)

def format_timestamp(seconds):
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = seconds % 60
    return f"{h:02d}:{m:02d}:{s:06.3f}"

# ---------------- RESUME LOGIC ----------------
resume_sec = ts_to_seconds(START_TIME)

if os.path.exists(CSV_FILE):
    with open(CSV_FILE, "r", encoding="utf-8") as f:
        rows = list(csv.reader(f))
        if len(rows) > 1:
            last_ts = rows[-1][0]
            resume_sec = ts_to_seconds(last_ts)
            print(f"Found existing CSV file: {CSV_FILE}")
            print(f"Resuming last location of: {last_ts}")
        else:
            print("CSV exists but contains no detections, starting fresh.")
else:
    print("No CSV found. Starting from beginning.")

# ---------------- VIDEO SETUP ----------------
cap = cv2.VideoCapture(VIDEO_FILE)
fps = cap.get(cv2.CAP_PROP_FPS)

cap.set(cv2.CAP_PROP_POS_MSEC, resume_sec * 1000)

last_roi = None
frame_number = int(resume_sec * fps)

# ---------------- JSON STREAMING SETUP ----------------
json_file = open(JSON_FILE, "a", encoding="utf-8")
if os.path.getsize(JSON_FILE) == 0:
    json_file.write("[\n")
    first_json = True
else:
    json_file.seek(0, os.SEEK_END)
    first_json = False

# ---------------- CSV SETUP ----------------
csv_exists = os.path.exists(CSV_FILE)
csv_file = open(CSV_FILE, "a", newline="", encoding="utf-8")
writer = csv.writer(csv_file)

if not csv_exists:
    writer.writerow(["timestamp", "ocr_text", "confidence"])

# ---------------- MAIN LOOP ----------------
while True:
    ret, frame = cap.read()
    if not ret:
        break

    current_sec = resume_sec + ((frame_number - int(resume_sec * fps)) / fps)
    if current_sec > ts_to_seconds(END_TIME):
        break

    roi = frame[Y0:Y1, X0:X1]

    changed = True
    if last_roi is not None:
        roi_gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
        last_gray = cv2.cvtColor(last_roi, cv2.COLOR_BGR2GRAY)
        roi_gray = cv2.resize(roi_gray, (last_gray.shape[1], last_gray.shape[0]))
        score = ssim(roi_gray, last_gray)
        changed = score < 0.92

    if changed:
        # OCR
        data = pytesseract.image_to_data(roi, config=TESS_CFG, output_type=Output.DICT)
        text = pytesseract.image_to_string(roi, config=TESS_CFG).strip()

        confs = []
        for c in data["conf"]:
            try:
                ci = int(c)
                if ci > 0:
                    confs.append(ci)
            except:
                pass
        avg_conf = sum(confs)/len(confs) if confs else 0.0

        last_roi = roi.copy()

        # Annotated frame
        annotated = frame.copy()
        cv2.rectangle(annotated, (X0, Y0), (X1, Y1), (0,255,0), 3)

        if text:
            y0 = 40
            for line in text.split("\n"):
                cv2.putText(annotated, line, (50, y0),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.0,
                            (0,255,0), 2, cv2.LINE_AA)
                y0 += 40

        # Timestamp in upper-right
        ts_str = format_timestamp(current_sec)
        (tw, th), _ = cv2.getTextSize(ts_str, cv2.FONT_HERSHEY_SIMPLEX, 1.2, 3)
        x = annotated.shape[1] - tw - 50
        y = 50 + th
        cv2.putText(annotated, ts_str, (x, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0,255,255), 3, cv2.LINE_AA)

        # Save annotated frame
        ann_path = os.path.join(ANNOTATED_DIR, f"{frame_number:06d}_annotated.jpg")
        cv2.imwrite(ann_path, annotated)

        # CSV write
        writer.writerow([ts_str, text, f"{avg_conf:.2f}"])
        csv_file.flush()

        # JSON write
        entry = {
            "index": str(frame_number),
            "timestamp": ts_str,
            "label": "frame",
            "ocr_text": text,
            "confidence": f"{avg_conf:.2f}"
        }

        if not first_json:
            json_file.write(",\n")
        first_json = False

        json_file.write(json.dumps(entry, indent=2))
        json_file.flush()
        os.fsync(json_file.fileno())

        # ---------------- TERMINAL STATUS UPDATE ----------------
        print(f"[DETECTION] {ts_str} — \"{text}\"")

    frame_number += 1

# Close JSON array
json_file.write("\n]\n")
json_file.close()
csv_file.close()
cap.release()

print("Done.")
